<a href="https://colab.research.google.com/github/Marlon-Sbardelatti/machine-learning/blob/master/trabalho-03/naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


> **Equipe**: 01<br>
> **Estudantes**: Cristina Siewert Jansen, Marlon Sbardelatti e Sofia Sousa Lindner



# PARTE 1: Algoritmo Naïve Bayes

Nesta primeira parte do Trabalho você irá aplicar o algoritmo de Naïve Bayes na base de dados de risco de crédito discutida em aula. Para isso você deve primeiramente importar as bibliotecas necessárias.

In [86]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [87]:
# importe a base de dados de risco de crédito e nomeie com: dataset_risco_credito
dataset_risco_credito = pd.read_csv('dataset_risco_credito.csv')

In [88]:
# imprima a base de dados para uma primeira avaliação dos dados
dataset_risco_credito

,historia,divida,garantias,renda,risco
0,ruim,alta,nenhuma,0_15,alto
1,desconhecida,alta,nenhuma,15_35,alto
2,desconhecida,baixa,nenhuma,15_35,moderado
3,desconhecida,baixa,nenhuma,acima_35,alto
4,desconhecida,baixa,nenhuma,acima_35,baixo
5,desconhecida,baixa,adequada,acima_35,baixo
6,ruim,baixa,nenhuma,0_15,alto
7,ruim,baixa,adequada,acima_35,moderado
8,boa,baixa,nenhuma,acima_35,baixo
9,boa,alta,adequada,acima_35,baixo


# 1 - Pré-Processamento dos dados

a) DIVISÃO DA BASE DE DADOS

Separe a base de dados dataset_risco_credito em:
 - variável x, com nome: X_risco_credito
 - classe y, com nome: y_risco_credito

DICA: você pode utilizar .iloc para selecionar as colunas da matriz e .values para converter para o numpy array.

In [89]:
X_risco_credito = dataset_risco_credito.iloc[:, 0:4].values
y_risco_credito = dataset_risco_credito.iloc[:, 4].values

b) APLICAR LABEL ENCODER

Perceba que seus dados possuem atributos categóricos (string). Porém, para aplicar esses dados em um algoritmo de aprendizado você precisa transformá-lo em atributo numérico.

Como você pode resolver isso?

DICA: Veja o que é e como aplicar o Label Enconder em: https://youtu.be/nLKEkBAbpQo

In [90]:
# Apresente o resultado do label enconder
from sklearn.preprocessing import LabelEncoder

label_encoder_historia = LabelEncoder()
label_encoder_divida = LabelEncoder()
label_encoder_garantia = LabelEncoder()
label_encoder_renda = LabelEncoder()
label_encoder_risco = LabelEncoder()

historia = label_encoder_historia.fit_transform(X_risco_credito[:, 0])
divida = label_encoder_divida.fit_transform(X_risco_credito[:, 1])
garantia = label_encoder_garantia.fit_transform(X_risco_credito[:, 2])
renda = label_encoder_renda.fit_transform(X_risco_credito[:, 3])

y_risco_credito = label_encoder_risco.fit_transform(y_risco_credito)
X_risco_credito = np.column_stack((historia, divida, garantia, renda))

print('História', historia) # (0) Boa | (1) Desconhecida | (2) Ruim
print('Dívida', divida) # (0) Alta | (1) Baixa
print('Garantia', garantia) # (0) Adequada | (1) Nenhuma
print('Renda',renda) # (0) 0 a 15 | (1) 15 a 35 | (2) Acima de 35
print('Risco', y_risco_credito) # (0) Alto | (1) Baixo | (2) Moderado

História [2 1 1 1 1 1 2 2 0 0 0 0 0 2]
Dívida [0 0 1 1 1 1 1 1 1 0 0 0 0 0]
Garantia [1 1 1 1 1 0 1 0 1 0 1 1 1 1]
Renda [0 1 1 2 2 2 0 2 2 2 0 1 2 1]
Risco [0 0 2 0 1 1 0 2 1 1 0 2 1 0]


c) SALVAR O ARQUIVO PRÉ-PROCESSADO

In [91]:
# como salvar o arquivo:
import pickle
with open('risco_credito.pkl', 'wb') as f:
  pickle.dump([X_risco_credito, y_risco_credito], f)

# 2 - Algoritmo Naïve Bayes

In [92]:
# importar da biblioteca sklearn o pacote Nayve Bayes
# utilizamos a distribuição estatística Gaussiana (classe GaussianNB) ou distribuição normal pois é mais usado para problemas genéricos
from sklearn.naive_bayes import GaussianNB

In [93]:
# Criar o objeto Nayve Bayes
naiveb_risco_credito = GaussianNB()

a) TREINAR O ALGORITMO

Para treinar o algoritmo, você deve gerar a tabela de probabilidades. Para isso, você pode utilizar **.fit** para gerar a tabela.

DICA: O 1º parametro são os atributos/características (x) e o 2º parametro é a classe (y).

OBS: Não se preocupe, o algoritmo faz a correção laplaciana automaticamente :) .

In [94]:
naiveb_risco_credito.fit(X = X_risco_credito, y = y_risco_credito)

GaussianNB()

b) FAZER A PREVISÃO

Utilize **.predict** para fazer a previsão realizada no exemplo em sala.

i) história boa, dívida alta, garantia nenhuma, renda > 35

ii) história ruim, dívida alta, garantia adequada, renda < 15

Verifique nos slides se seu resultado está correto!

In [95]:
# utilize o atributo .classes_ para mostrar as classes utilizadas pelo algoritmo
naiveb_risco_credito.classes_

array([0, 1, 2])

In [96]:
# utilize .class_count_ para contar quantos registros tem em cada classe
naiveb_risco_credito.class_count_

array([6., 5., 3.])

In [97]:
# i) história boa, dívida alta, garantia nenhuma, renda > 35
X_risco_credito_1 = np.array([[0, 0, 1, 2]])
previsao_credito_1 = naiveb_risco_credito.predict(X_risco_credito_1)

print(label_encoder_risco.inverse_transform(previsao_credito_1))

['baixo']


**i) Prova real com base no slide**

---

P(Alto) = 6/14 * 1/6 * 4/6 * 6/6 * 1/6 = 0.0079

P(Moderado) = 3/14 * 1/3 * 1/3 * 2/3 * 1/3 = 0.0052

P(Baixo) = 5/14 * 3/5 * 2/5 * 3/5 * 5/5 = 0.0514

Soma = 0.0079 + 0.0052 + 0.0514 = 0.0645

<br>

**P(Alto)** = 0.0079 / 0.0645 * 100 = 12,25%

**P(Moderado)** = 0.0052 / 0.0645 * 100 = 8,06%

**P(Baixo)** = 0.0514 / 0.0645 * 100 = 79,69%


In [98]:
# ii) história ruim, dívida alta, garantia adequada, renda < 15
X_risco_credito_2 = np.array([[2, 0, 0, 0]])
previsao_credito_2 = naiveb_risco_credito.predict(X_risco_credito_2)

print(label_encoder_risco.inverse_transform(previsao_credito_2))

['moderado']


**ii) Prova real com base no slide**

---

P(Alto) = 7/15 * 3/7 * 4/7 * 1/7 * 3/7 = 0.0070

P(Moderado) = 4/15 * 1/4 * 1/4 * 1/4 * 1/4 = 0.0010

P(Baixo) = 6/15 * 1/6 * 2/6 * 2/6 * 1/6 = 0.0012

Soma = 0.0070 + 0.0010 + 0.0012 = 0.0092

<br>

**P(Alto)** = 0.0070 / 0.0092 = 0.7609 = 76,09%

**P(Moderado)** = 0.0010 / 0.0092 = 0.1087 = 10,87%

**P(Baixo)** = 0.0012 / 0.0092 = 0.1304 = 13,04%
